# Member 1: Data Engineering & Common Pipeline
### Enterprise-Level Project: Causal Inference & Synthetic Control Engine for Algorithmic Pricing
**Organization:** Zaalima Development — Enterprise Data Analytics  
**Author:** Member 1 (Data Engineering Lead)  
**Downstream Ingestion By:** Member 2 (DAG), Member 3 (Synthetic Control), Member 4 (Double ML)

---

## 1. Responsibilities & Objectives
As **Member 1**, this notebook delivers the data foundation for the entire causal inference engine:
1. **Retail Transaction Dataset Collection & Ingestion:** Simulating realistic transaction telemetry across 10 retail locations over 52 weeks.
2. **Data Cleaning & Quality Assurance:** Detecting and removing duplicates, filtering corrupt records (negative prices/returns), and imputing missing sensor values.
3. **Store x Product x Week Panel Construction:** Aggregating daily transactions into a balanced panel suitable for econometric estimators.
4. **Exploratory Data Analysis (EDA):** Validating pricing trends, parallel pre-treatment trajectories, and confounder correlations.
5. **Downstream Handover:** Generating the master `.csv` and styled `.xlsx` deliverables for Members 2, 3, and 4.

## 2. Environment Setup & Library Imports

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as plt_sns

# Ensure project root in python path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

print(f"Project root configured: {project_root}")

## 3. Step 1: Raw Data Ingestion & Quality Inspection

In [ ]:
raw_csv_path = os.path.join(project_root, "data", "raw", "retail_transactions_raw.csv")
df_raw = pd.read_csv(raw_csv_path)

print("=== Raw Data Overview ===")
print(f"Total rows: {len(df_raw):,}")
print(f"Total columns: {len(df_raw.columns)}")
print("\nMissing values by column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])

dup_count = df_raw.duplicated(subset=["date", "store_id", "product_id"]).sum()
print(f"\nDuplicate natural keys detected: {dup_count}")
df_raw.head(5)

## 4. Step 2: Data Cleaning & QA Execution

In [ ]:
from src.data_pipeline.clean_dataset import clean_retail_data

cleaned_csv_path = os.path.join(project_root, "data", "processed", "retail_transactions_cleaned.csv")
report_path = os.path.join(project_root, "reports", "data_quality_report.md")

df_cleaned = clean_retail_data(raw_csv_path, cleaned_csv_path, report_path)
print(f"Post-cleaning rows retained: {len(df_cleaned):,} (Retention: {round(len(df_cleaned)/len(df_raw)*100, 2)}%)")

## 5. Step 3: Panel Dataset Construction (Store x Product x Week)

In [ ]:
from src.data_pipeline.build_panel import build_store_week_panel

panel_csv_path = os.path.join(project_root, "data", "processed", "store_product_week_panel.csv")
df_panel = build_store_week_panel(cleaned_csv_path, panel_csv_path)

print(f"Panel shape: {df_panel.shape}")
df_panel.head(5)

## 6. Step 4: Exploratory Data Analysis & Causal Validation

In [ ]:
from src.data_pipeline.eda_analysis import run_eda

charts_dir = os.path.join(project_root, "reports", "eda_figures")
run_eda(panel_csv_path, charts_dir)

# Display generated figures
from IPython.display import Image, display
p1 = os.path.join(charts_dir, "01_price_trajectories_intervention.png")
p3 = os.path.join(charts_dir, "03_parallel_trends_validation.png")

print("Figure 1: Price Intervention Trajectories (Store 1 vs Donor Pool)")
if os.path.exists(p1):
    display(Image(filename=p1, width=800))

print("Figure 3: Pre-Treatment Parallel Trends Validation (Weeks 1 to 35)")
if os.path.exists(p3):
    display(Image(filename=p3, width=800))

## 7. Step 5: Multi-Tab Master Excel Workbook Generation

In [ ]:
from src.data_pipeline.export_excel import create_styled_excel

xlsx_path = os.path.join(project_root, "data", "processed", "store_product_week_panel.xlsx")
create_styled_excel(panel_csv_path, xlsx_path)
print(f"Master Excel generated for team distribution: {xlsx_path}")